In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

### Data and Model

In [2]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=10.33s)


In [3]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])

dm = MTDataModule(_config, dist=False)
model = METERTransformerSS(_config)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.bias']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### Data Class

In [22]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, split='', max_bb = 75):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []
        

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            ref = self.refer.Refs[ref_id]
            for sent_id in ref['sent_ids']:
                sent_ids.append(sent_id)
        return sent_ids

    def __getitem__(self, index):
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = self.refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        
        sub_images = []
        for obj in objs:
            try:
                x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            except ValueError:
                print(f'ValueError at setence id: {sent_id}')
                self.duds.append(sent_id)
                break
                
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)      
            
        text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
        text_labels = [[-100 for i in range(40)]]

        ids = [text_ids for i in range(num_sub_images)]
        masks = [text_masks for _ in range(num_sub_images)]
        labels = [text_labels for i in range(num_sub_images)]
            
        return_dict = {
            'ann_id' : ann_id,
            'image' : sub_images,
            'obj_ids' : obj_ids,
            'num_bb' : num_sub_images,
            'sent_id' : sent_id,
            'text' : sent['sent'],
            'text_ids' : torch.tensor(ids),
            'text_labels' : torch.tensor(labels),
            'text_masks' : torch.tensor(masks)
        }  

        return return_dict
    
    def collate_fn(self, batch):
        max_bb = max([example['num_bb'] for example in batch])
        ann_id =  [example['ann_id'] for example in batch]

In [26]:
batch = []
for i in range(10):
    batch.append(ds[i])
batch

[{'ann_id': 1719310,
  'image': [tensor([[[[0.0549, 0.0549, 0.0549,  ..., 0.0078, 0.0000, 0.0000],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0039, 0.0039],
             ...,
             [0.2824, 0.2588, 0.2157,  ..., 0.6784, 0.7176, 0.7412],
             [0.2745, 0.2549, 0.2235,  ..., 0.8588, 0.8627, 0.8627],
             [0.2745, 0.2588, 0.2353,  ..., 0.6941, 0.6745, 0.6627]],
   
            [[0.0510, 0.0510, 0.0510,  ..., 0.0118, 0.0078, 0.0039],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0000, 0.0000],
             ...,
             [0.4314, 0.3922, 0.3216,  ..., 0.7804, 0.8196, 0.8471],
             [0.4157, 0.3843, 0.3255,  ..., 0.8863, 0.8706, 0.8627],
             [0.4157, 0.3882, 0.3373,  ..., 0.6471, 0.6078, 0.5843]],
   
            [[0.0314, 0.0314, 0.0314,  ..., 0.0039, 0.0039, 0.0039],
             [0.0353, 0.035

In [28]:
max_bb = max([example['num_bb'] for example in batch])
max_bb

33

In [33]:
batch[0]['image'].__len__()

33

In [25]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 10

epochs = 1

ds = RefcocoDataset(refer, tokenizer)

In [18]:
ds[0]

{'ann_id': 1719310,
 'image': [tensor([[[[0.0549, 0.0549, 0.0549,  ..., 0.0078, 0.0000, 0.0000],
            [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
            [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0039, 0.0039],
            ...,
            [0.2824, 0.2588, 0.2157,  ..., 0.6784, 0.7176, 0.7412],
            [0.2745, 0.2549, 0.2235,  ..., 0.8588, 0.8627, 0.8627],
            [0.2745, 0.2588, 0.2353,  ..., 0.6941, 0.6745, 0.6627]],
  
           [[0.0510, 0.0510, 0.0510,  ..., 0.0118, 0.0078, 0.0039],
            [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
            [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0000, 0.0000],
            ...,
            [0.4314, 0.3922, 0.3216,  ..., 0.7804, 0.8196, 0.8471],
            [0.4157, 0.3843, 0.3255,  ..., 0.8863, 0.8706, 0.8627],
            [0.4157, 0.3882, 0.3373,  ..., 0.6471, 0.6078, 0.5843]],
  
           [[0.0314, 0.0314, 0.0314,  ..., 0.0039, 0.0039, 0.0039],
            [0.0353, 0.0353, 0.0353,  ..., 0.

In [19]:
ds = RefcocoDataset(refer, tokenizer)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = torch.utils.data.DataLoader(ds, **train_params)


In [20]:
for i, data in enumerate(training_loader):
    if i ==1:
        break
data['ann_id'].shape

RuntimeError: each element in list of batch should be of equal size